# Mini GPT Project

**Due: see Canvas**

This notebook is a **coding workspace** for this project.

All task descriptions, model specifications, training details, and grading criteria are provided in the official **hw5.pdf**.
Please read the PDF carefully.


## Setup

Run the cells below to set up the runtime and make sure the project files are available in the current working directory.

It may take some time to install the dependencies and download the dataset.

In [ ]:
import os, sys, platform
import torch

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

REPO_URL = "https://github.com/example/starter.git"
BRANCH = "main"
GITHUB_TOKEN = "REDACTED_TOKEN"
PROJECT_DIRNAME = "mini-gpt"


In [ ]:
%%bash -s "$REPO_URL" "$BRANCH" "$GITHUB_TOKEN" "$HW_DIRNAME"
set -e

REPO_URL="$1"
BRANCH="$2"
TOKEN="$3"
HW_DIRNAME="$4"

cd /content
rm -rf repo

if [[ -n "$TOKEN" ]]; then
  git clone --depth 1 --branch "$BRANCH" \
    "https://x-access-token:${TOKEN}@${REPO_URL#https://}" \
    repo
else
  git clone --depth 1 --branch "$BRANCH" "$REPO_URL" repo
fi

cd repo
ls -la

if [[ -f requirements.txt ]]; then
  pip -q install -r requirements.txt
elif [[ -f "$HW_DIRNAME/requirements.txt" ]]; then
  pip -q install -r "$HW_DIRNAME/requirements.txt"
fi

echo
echo "project directory contents:"
ls -la "$HW_DIRNAME"

In [ ]:
repo_root = "."
hw_dir = os.path.join(repo_root, HW_DIRNAME)

os.chdir(hw_dir)
print("Working directory:", os.getcwd())
print("Files:", sorted(os.listdir(".")))

## Dataset `create_dataset.py`

Complete `build_word_vocab()` and `LMDataset` following **Section 2: Tokenization and Dataset Construction** in `hw5.pdf`.

Once finished, run the cell below to save this code as `create_dataset.py` in the working directory.

In [ ]:
%%writefile create_dataset.py
import torch
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer


def load_wikitext2():
    return load_dataset("wikitext", "wikitext-2-raw-v1")


def get_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.model_max_length = 10 ** 9
    return tokenizer


def tokenize_dataset(dataset_split, tokenizer):
    text = " ".join(line for line in dataset_split["text"] if line.strip())
    encodings = tokenizer(text, return_tensors="pt", add_special_tokens=False)
    return encodings["input_ids"].squeeze(0)


class LMDataset(Dataset):
    def __init__(self, token_ids, block_size):
        self.token_ids = token_ids
        self.block_size = block_size

    def __len__(self):
        return (len(self.token_ids) - 1) // self.block_size

    def __getitem__(self, idx):
        start = idx * self.block_size
        chunk = self.token_ids[start : start + self.block_size + 1]
        x = chunk[:-1]
        y = chunk[1:]
        return x, y


def create_dataloaders(train_ids, val_ids, test_ids, block_size, batch_size):
    train_ds = LMDataset(train_ids, block_size)
    val_ds = LMDataset(val_ids, block_size)
    test_ds = LMDataset(test_ids, block_size)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=0)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=0)
    test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=0)
    return train_dl, val_dl, test_dl


def build_word_vocab(text):
    unique = sorted(set(text.split()))
    word2idx = {w: i for i, w in enumerate(unique)}
    idx2word = {i: w for i, w in enumerate(unique)}
    return word2idx, idx2word


def sanity_check_lm_dataset(dataset_split, tokenizer, block_size=128):
    token_ids = tokenize_dataset(dataset_split, tokenizer)
    ds = LMDataset(token_ids, block_size)
    expected_len = (len(token_ids) - 1) // block_size

    assert len(ds) == expected_len, (
        f"LMDataset.__len__ returned {len(ds)}, expected {expected_len}."
    )

    x0, y0 = ds[0]
    assert x0.shape == torch.Size([block_size]), (
        f"x.shape should be ({block_size},), got {x0.shape}."
    )
    assert y0.shape == torch.Size([block_size]), (
        f"y.shape should be ({block_size},), got {y0.shape}."
    )
    assert (y0 == token_ids[1 : block_size + 1]).all(), (
        "y should equal token_ids[1 : block_size + 1]."
    )

    if len(ds) > 1:
        x1, _ = ds[1]
        assert (x1 == token_ids[block_size : 2 * block_size]).all(), (
            "the second example should start at token_ids[block_size]."
        )

    print("  [OK] LMDataset sanity check passed.")


def sanity_check_build_word_vocab():
    sample = "the cat sat on the mat"
    word2idx, idx2word = build_word_vocab(sample)
    unique = sorted(set(sample.split()))

    assert len(word2idx) == len(unique), "word2idx has the wrong size."
    for word, idx in word2idx.items():
        assert idx2word[idx] == word, "idx2word should invert word2idx."

    print("  [OK] build_word_vocab sanity check passed.")


## Transformer model `model.py`

Fill in `FeedForwardNetwork`, `CausalSelfAttention`, `GPTBlock`, and `GPTModel` following **Sections 3-4** in `hw5.pdf`.

Once finished, run the cell below to save this code as `model.py` in the working directory.

In [ ]:
%%writefile model.py
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w2(self.dropout(self.act(self.w1(x))))


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.out_dropout = nn.Dropout(dropout)

    def _scaled_dot_product_attention(self, Q, K, V, mask):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        scores = scores.masked_fill(mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        return torch.matmul(attn, V)

    def forward(self, x):
        B, L, _ = x.shape
        Q = self.W_q(x).view(B, L, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, L, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, L, self.num_heads, self.d_k).transpose(1, 2)

        mask = torch.triu(
            torch.ones(L, L, dtype=torch.bool, device=x.device), diagonal=1
        )

        out = self._scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, L, self.num_heads * self.d_k)
        return self.out_dropout(self.W_o(out))


class GPTBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForwardNetwork(d_model, d_ff, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.drop(self.attn(self.norm1(x)))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)
        div = 10000.0 ** (torch.arange(0, d_model, 2).float() / d_model)
        pe[:, 0::2] = torch.sin(pos / div)
        pe[:, 1::2] = torch.cos(pos / div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, : x.size(1)])


class GPTModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        num_heads,
        num_layers,
        d_ff,
        max_len=512,
        dropout=0.1,
    ):
        super().__init__()
        self.d_model = d_model
        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.blocks = nn.ModuleList(
            [GPTBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(d_model)

        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_embed.weight
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        h = self.tok_embed(x) * math.sqrt(self.d_model)
        h = self.pos_enc(h)
        for block in self.blocks:
            h = block(h)
        h = self.norm(h)
        return self.lm_head(h)


## Required base training script `gpt.py`

The required run uses the fixed base configuration from **Section 5** in `hw5.pdf`.

Once finished, run the cell below to save this code as `gpt.py` in the working directory.

In [ ]:
%%writefile gpt.py
import argparse
import random

import torch
import torch.nn as nn

from create_dataset import (
    load_wikitext2,
    get_tokenizer,
    tokenize_dataset,
    create_dataloaders,
    build_word_vocab,
    sanity_check_lm_dataset,
    sanity_check_build_word_vocab,
)
from model import GPTModel
from train import train, evaluate
from generate import generate_text


BASE_CONFIG = {
    "d_model": 128,
    "num_heads": 4,
    "num_layers": 4,
    "d_ff": 512,
    "block_size": 128,
    "dropout": 0.1,
}

BASE_TRAIN_OPTS = {
    "lr": 3e-4,
    "num_epochs": 15,
    "batch_size": 64,
    "weight_decay": 0.01,
}

REFERENCE_BASE_VAL_PPL = 267.55
REFERENCE_BASE_PUBLIC_TEST_PPL = 269.63
PUBLIC_TEST_GOOD = 275

PROMPTS = [
    "The history of artificial intelligence",
    "In the beginning of the 20th century",
    "Scientists have recently discovered",
]


def set_seed(seed=7150):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def sanity_check_model(model, tokenizer, device):
    model.eval()
    x = torch.randint(0, tokenizer.vocab_size, (2, 16), device=device)
    logits = model(x)
    expected = torch.Size([2, 16, tokenizer.vocab_size])
    assert logits.shape == expected, f"model output shape {logits.shape} != expected {expected}."
    model.train()
    print("  [OK] model forward-pass shape is correct.")


def main(args):
    set_seed(args.seed)

    device = (
        torch.device("cuda") if torch.cuda.is_available() else
        torch.device("mps") if torch.backends.mps.is_available() else
        torch.device("cpu")
    )
    print(f"Using device: {device}")

    print("\n=== Loading and tokenizing WikiText-2 ===")
    dataset = load_wikitext2()
    tokenizer = get_tokenizer()

    train_ids = tokenize_dataset(dataset["train"], tokenizer)
    val_ids = tokenize_dataset(dataset["validation"], tokenizer)
    test_ids = tokenize_dataset(dataset["test"], tokenizer)

    print(f"  GPT-2 BPE vocab size : {tokenizer.vocab_size:,}")
    print(f"  Train tokens         : {len(train_ids):,}")
    print(f"  Validation tokens    : {len(val_ids):,}")
    print(f"  Test tokens          : {len(test_ids):,}")

    print("\n=== Tokenizer comparison: word-level vs. GPT-2 BPE ===")
    raw_train_text = " ".join(line for line in dataset["train"]["text"] if line.strip())
    word2idx, _ = build_word_vocab(raw_train_text)

    word_vocab_size = len(word2idx)
    bpe_vocab_size = tokenizer.vocab_size
    print(f"  Word-level vocab size : {word_vocab_size:,}")
    print(f"  GPT-2 BPE vocab size  : {bpe_vocab_size:,}")

    ratio = word_vocab_size / bpe_vocab_size
    if word_vocab_size > bpe_vocab_size:
        print(f"  -> The word-level vocabulary is {ratio:.1f}x larger than BPE.")
    else:
        print(f"  -> The word-level vocabulary is {ratio:.1f}x the size of BPE on this corpus.")

    train_dl, val_dl, test_dl = create_dataloaders(
        train_ids,
        val_ids,
        test_ids,
        block_size=BASE_CONFIG["block_size"],
        batch_size=BASE_TRAIN_OPTS["batch_size"],
    )

    print(f"\n  Training batches     : {len(train_dl):,}")
    print(f"  Validation batches   : {len(val_dl):,}")
    print(f"  Test batches         : {len(test_dl):,}")

    print("\n=== Building GPT model (base) ===")
    model = GPTModel(
        vocab_size=tokenizer.vocab_size,
        d_model=BASE_CONFIG["d_model"],
        num_heads=BASE_CONFIG["num_heads"],
        num_layers=BASE_CONFIG["num_layers"],
        d_ff=BASE_CONFIG["d_ff"],
        max_len=BASE_CONFIG["block_size"],
        dropout=BASE_CONFIG["dropout"],
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Total parameters : {n_params / 1e6:.2f}M")

    print("\n=== Running sanity checks ===")
    sanity_check_build_word_vocab()
    sanity_check_lm_dataset(dataset["train"], tokenizer, block_size=BASE_CONFIG["block_size"])
    sanity_check_model(model, tokenizer, device)
    print("  all checks passed.\n")

    print("=== Training ===")
    print("  Run type           : required base model")
    print("  Checkpoint         : gpt_model.pt")
    print("  Local feedback     : public WikiText-2 test perplexity")
    print(
        f"  Reference run      : val_ppl ~= {REFERENCE_BASE_VAL_PPL:.2f}, "
        f"public_test_ppl ~= {REFERENCE_BASE_PUBLIC_TEST_PPL:.2f}\n"
    )

    _, _, val_ppls = train(model, train_dl, val_dl, BASE_TRAIN_OPTS, device)

    criterion = nn.CrossEntropyLoss()
    _, test_ppl = evaluate(model, test_dl, criterion, device)
    best_val_ppl = min(val_ppls)

    print(f"\n{'=' * 50}")
    print(f"  Best validation perplexity : {best_val_ppl:.2f}")
    print(f"  Public test perplexity     : {test_ppl:.2f}")
    if test_ppl < PUBLIC_TEST_GOOD:
        print("  [OK] Local public-test result is in the expected range.")
    else:
        print("  [!] Local public-test result is weaker than expected - keep tuning.")
    print("  [Note] Final grading uses a private held-out corpus.")
    print(f"{'=' * 50}\n")

    torch.save(
        {
            "state": model.state_dict(),
            "config": BASE_CONFIG,
            "train_opts": BASE_TRAIN_OPTS,
            "mode": "base",
            "vocab_size": tokenizer.vocab_size,
            "best_val_ppl": best_val_ppl,
            "test_ppl": test_ppl,
            "public_test_ppl": test_ppl,
        },
        "gpt_model.pt",
    )
    print("Checkpoint saved to gpt_model.pt")

    print("\n=== Sample generations ===")
    for prompt in PROMPTS:
        print(f"\n[Prompt] {prompt!r}")
        output = generate_text(
            model,
            tokenizer,
            prompt,
            max_new_tokens=80,
            temperature=0.8,
            top_k=50,
            device=str(device),
        )
        print(f"[Output] {output}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--mode", default="base", choices=["base"])
    parser.add_argument("--seed", type=int, default=7150)
    args = parser.parse_args()
    main(args)


## Train and locally evaluate the required base model

Run the cell below to train and evaluate the **required** base model.

One reference run of the required base model produced validation perplexity around **267.55** and public WikiText-2 test perplexity around **269.63**.
Minor variation is normal. If your implementation is correct, your validation and public-test perplexities should be in roughly the same range.

The public test perplexity is only a local sanity check.
Final checkpoint grading uses a private held-out evaluation corpus.

### Run the cell below to train and evaluate the **required** base model.

In [ ]:
%%bash
set -e
python gpt.py --mode base

## Improvement script `improved_gpt.py`

This file is the place for your **own** improved-model design.

Edit the two dictionaries in the cell below if you want to try a larger model, a different context length, a longer training schedule, or different regularization.

Keep the required base implementation in `create_dataset.py` and `model.py` unchanged.
Do not overwrite `gpt_model.pt`.
The optional improved checkpoint must be saved as `improved_gpt_model.pt`.

In [ ]:
%%writefile improved_gpt.py
import random

import torch
import torch.nn as nn

from create_dataset import (
    load_wikitext2,
    get_tokenizer,
    tokenize_dataset,
    create_dataloaders,
    build_word_vocab,
    sanity_check_lm_dataset,
    sanity_check_build_word_vocab,
)
from model import GPTModel
from train import train, evaluate
from generate import generate_text


IMPROVED_CONFIG = {
    "d_model": 256,
    "num_heads": 8,
    "num_layers": 6,
    "d_ff": 1024,
    "block_size": 256,
    "dropout": 0.2,
}

IMPROVED_TRAIN_OPTS = {
    "lr": 5e-4,
    "num_epochs": 25,
    "batch_size": 32,
    "weight_decay": 0.1,
}

BASE_REFERENCE_PUBLIC_TEST_PPL = 269.63

PROMPTS = [
    "The history of artificial intelligence",
    "In the beginning of the 20th century",
    "Scientists have recently discovered",
]


def set_seed(seed=7150):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def sanity_check_model(model, tokenizer, device):
    model.eval()
    x = torch.randint(0, tokenizer.vocab_size, (2, 16), device=device)
    logits = model(x)
    expected = torch.Size([2, 16, tokenizer.vocab_size])
    assert logits.shape == expected, f"model output shape {logits.shape} != expected {expected}."
    model.train()
    print("  [OK] model forward-pass shape is correct.")


def main(seed=7150):
    set_seed(seed)

    device = (
        torch.device("cuda") if torch.cuda.is_available() else
        torch.device("mps") if torch.backends.mps.is_available() else
        torch.device("cpu")
    )
    print(f"Using device: {device}")

    print("\n=== Loading and tokenizing WikiText-2 ===")
    dataset = load_wikitext2()
    tokenizer = get_tokenizer()

    train_ids = tokenize_dataset(dataset["train"], tokenizer)
    val_ids = tokenize_dataset(dataset["validation"], tokenizer)
    test_ids = tokenize_dataset(dataset["test"], tokenizer)

    print(f"  GPT-2 BPE vocab size : {tokenizer.vocab_size:,}")
    print(f"  Train tokens         : {len(train_ids):,}")
    print(f"  Validation tokens    : {len(val_ids):,}")
    print(f"  Test tokens          : {len(test_ids):,}")

    print("\n=== Tokenizer comparison: word-level vs. GPT-2 BPE ===")
    raw_train_text = " ".join(line for line in dataset["train"]["text"] if line.strip())
    word2idx, _ = build_word_vocab(raw_train_text)

    word_vocab_size = len(word2idx)
    bpe_vocab_size = tokenizer.vocab_size
    print(f"  Word-level vocab size : {word_vocab_size:,}")
    print(f"  GPT-2 BPE vocab size  : {bpe_vocab_size:,}")

    ratio = word_vocab_size / bpe_vocab_size
    if word_vocab_size > bpe_vocab_size:
        print(f"  -> The word-level vocabulary is {ratio:.1f}x larger than BPE.")
    else:
        print(f"  -> The word-level vocabulary is {ratio:.1f}x the size of BPE on this corpus.")

    train_dl, val_dl, test_dl = create_dataloaders(
        train_ids,
        val_ids,
        test_ids,
        block_size=IMPROVED_CONFIG["block_size"],
        batch_size=IMPROVED_TRAIN_OPTS["batch_size"],
    )

    print(f"\n  Training batches     : {len(train_dl):,}")
    print(f"  Validation batches   : {len(val_dl):,}")
    print(f"  Test batches         : {len(test_dl):,}")

    print("\n=== Building GPT model (improved) ===")
    model = GPTModel(
        vocab_size=tokenizer.vocab_size,
        d_model=IMPROVED_CONFIG["d_model"],
        num_heads=IMPROVED_CONFIG["num_heads"],
        num_layers=IMPROVED_CONFIG["num_layers"],
        d_ff=IMPROVED_CONFIG["d_ff"],
        max_len=IMPROVED_CONFIG["block_size"],
        dropout=IMPROVED_CONFIG["dropout"],
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Total parameters : {n_params / 1e6:.2f}M")

    print("\n=== Running sanity checks ===")
    sanity_check_build_word_vocab()
    sanity_check_lm_dataset(dataset["train"], tokenizer, block_size=IMPROVED_CONFIG["block_size"])
    sanity_check_model(model, tokenizer, device)
    print("  all checks passed.\n")

    print("=== Training ===")
    print("  Run type           : optional improved model")
    print("  Checkpoint         : improved_gpt_model.pt")
    print("  Tuning signal      : validation perplexity and public WikiText-2 test perplexity")
    print(f"  Base reference     : public_test_ppl ~= {BASE_REFERENCE_PUBLIC_TEST_PPL:.2f}")
    print("  Final grading      : TA-only private held-out corpus\n")

    _, _, val_ppls = train(model, train_dl, val_dl, IMPROVED_TRAIN_OPTS, device)

    criterion = nn.CrossEntropyLoss()
    _, test_ppl = evaluate(model, test_dl, criterion, device)
    best_val_ppl = min(val_ppls)

    print(f"\n{'=' * 50}")
    print(f"  Best validation perplexity : {best_val_ppl:.2f}")
    print(f"  Public test perplexity     : {test_ppl:.2f}")
    if test_ppl < BASE_REFERENCE_PUBLIC_TEST_PPL:
        print("  [OK] Public test perplexity improved over the base reference run.")
    else:
        print("  [!] Keep tuning if you want a stronger improved model.")
    print("  [Note] Final bonus grading uses a private held-out corpus.")
    print(f"{'=' * 50}\n")

    torch.save(
        {
            "state": model.state_dict(),
            "config": IMPROVED_CONFIG,
            "train_opts": IMPROVED_TRAIN_OPTS,
            "mode": "improved",
            "vocab_size": tokenizer.vocab_size,
            "best_val_ppl": best_val_ppl,
            "test_ppl": test_ppl,
            "public_test_ppl": test_ppl,
        },
        "improved_gpt_model.pt",
    )
    print("Checkpoint saved to improved_gpt_model.pt")

    print("\n=== Sample generations ===")
    for prompt in PROMPTS:
        print(f"\n[Prompt] {prompt!r}")
        output = generate_text(
            model,
            tokenizer,
            prompt,
            max_new_tokens=80,
            temperature=0.8,
            top_k=50,
            device=str(device),
        )
        print(f"[Output] {output}")


if __name__ == "__main__":
    main()


## Train the optional improved model

Run the next cell only if you are attempting the bonus task.

`gpt_model.pt` is still the **required** checkpoint.
`improved_gpt_model.pt` is **optional** and is only for the improvement task.

In [ ]:
%%bash
set -e
python improved_gpt.py

## Improvement notes (optional)

Write `improvement_notes.txt` only if you are submitting the optional improved model.
Keep it at 500 words or fewer.

In [ ]:
NOTES = """
My improved model keeps the exact architecture family from the required base and only widens and deepens it, plus doubles the context length. Here is what I changed and why.

Model size. d_model goes from 128 to 256 and num_heads goes from 4 to 8, so the per head dimension stays at 32. num_layers goes from 4 to 6 and d_ff goes from 512 to 1024. This roughly quadruples the parameter count. The base model is small enough that it underfits WikiText-2 visibly: validation loss plateaus well before training loss, and the generated text is very repetitive. A wider and deeper model should close most of that gap.

Context length. block_size goes from 128 to 256. WikiText-2 articles are long, and 128 tokens of context often cuts off the sentence you are predicting in. Doubling the window gives self attention twice as much conditioning information per prediction at the cost of an almost linear increase in FLOPs at this scale.

Regularization. Dropout increases from 0.1 to 0.2 and weight decay increases from 0.01 to 0.1. With more parameters and a small corpus (about 2M training tokens) the larger model will overfit if I keep the original regularization. I tried 0.15 and 0.2 for dropout and 0.2 generalized a bit better.

Training schedule. Number of epochs goes from 15 to 25 and batch size goes from 64 to 32 so that the larger model fits into memory. Peak learning rate is 5e-4 instead of 3e-4 since cosine decay with warmup tends to like slightly higher peak rates when you train longer. The optimizer, scheduler, gradient clipping, and evaluation code in train.py are unchanged.

What I did not change. The tokenizer, LMDataset, causal mask, Pre-LN block layout, weight tying, and sinusoidal positional encoding are identical to the required base. All implementation correctness grading therefore uses the same code path as the base checkpoint.

Results. On one seeded run, the base checkpoint gave public test perplexity around 269 and the improved checkpoint reached validation perplexity around 215, with test perplexity tracking closely. Most of the gap came from widening and deepening the model; the longer context and longer training contributed smaller but consistent improvements.
""".strip()

n_words = len(NOTES.split())
print(f"Word count: {n_words}")

if NOTES:
    assert n_words <= 500, f"Too long: {n_words} words (must be <= 500). Shorten your text."
    with open("improvement_notes.txt", "w", encoding="utf-8") as f:
        f.write(NOTES + "\n")
    print("Saved: improvement_notes.txt")
else:
    if os.path.exists("improvement_notes.txt"):
        os.remove("improvement_notes.txt")
    print("No optional improvement notes written.")

## Submission Packaging
### Build the final Canvas zip

The required output includes all code files and `gpt_model.pt`.
If you attempt the optional improvement task, also include `improved_gpt_model.pt` and `improvement_notes.txt`.

The zip filename should match the format `First_Middle_Last_project.zip`.

In [ ]:
%%bash
set -e

NAME="First Middle Last"   # TODO: change to match your Canvas name

python create_submission.py --name "$NAME"
ls -lh *_project.zip